In [0]:
#Import libraries
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, DateType, IntegerType
from pyspark.sql.functions import trim,col,length

In [0]:
# Read data bronze table and load to dataframe
df = spark.table("baraa_projectwork.bronze.crm_sales_details")


### Transformation

In [0]:
# Data Cleaning: Extracts only the integer value from the sls_ord_num column
df = df.withColumn("sls_ord_num", F.substring(col("sls_ord_num"), 3, F.length(col("sls_ord_num"))))

In [0]:
# Data Cleaning: Convert sls_ord_num from string to int
df = df.withColumn("sls_ord_num", F.col("sls_ord_num").cast("Integer"))
display(df)

In [0]:
# Date Conversion: Convert sls_order_dt, sls_ship_dt, sls_due_dt from integer to date
df = (
    df
    .withColumn(
        "sls_order_dt",
        F.when(
            (col("sls_order_dt") == 0) | (length(col("sls_order_dt")) != 8),
            None
        ).otherwise(F.to_date(col("sls_order_dt").cast("string"), "yyyyMMdd"))
    )
    .withColumn(
        "sls_ship_dt",
        F.when(
            (col("sls_ship_dt") == 0) | (length(col("sls_ship_dt")) != 8),
            None
        ).otherwise(F.to_date(col("sls_ship_dt").cast("string"), "yyyyMMdd"))
    )
    .withColumn(
        "sls_due_dt",
        F.when(
            (col("sls_due_dt") == 0) | (length(col("sls_due_dt")) != 8),
            None
        ).otherwise(F.to_date(col("sls_due_dt").cast("string"), "yyyyMMdd"))
    )
)

In [0]:
# Data Cleaning: Impute missing or invalid sls_price values
df = (
    df.withColumn(
        "sls_price",
        F.when(
            (col("sls_price").isNull()) | (col("sls_price") <= 0),  # If sls_price is null or <= 0
            F.when(
                col("sls_quantity") != 0,                            # If sls_quantity is not zero
                col("sls_sales") / col("sls_quantity")               # Calculate price as sales divided by quantity
            ).otherwise(None)                                       # Otherwise set to None
        ).otherwise(col("sls_price"))                               # Else keep original sls_price
    )
)

display(df)

### Rename Columns

In [0]:
RENAME_MAP = {
    "sls_ord_num": "order_number",
    "sls_prd_key": "product_number",
    "sls_cust_id": "customer_id",
    "sls_order_dt": "order_date",
    "sls_ship_dt": "ship_date",
    "sls_due_dt": "due_date",
    "sls_sales": "sales",
    "sls_quantity": "quantity",
    "sls_price": "price"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

### Writing to Silver table


In [0]:
df.limit(10).display()

In [0]:

df.write.mode("overwrite").format("delta").saveAsTable("baraa_projectwork.silver.crm_sales_details")

In [0]:
%sql
SELECT * FROM baraa_projectwork.silver.crm_sales_details LIMIT 10